In [ ]:
!apt-get install poppler-utils -y
!pip install pdf2image pillow opencv-python requests

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (252 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Archit Parag Ethics Updated.pdf to Archit Parag Ethics Updated.pdf


In [ ]:
import os
import json
import time
import requests
import cv2
import numpy as np
from pdf2image import convert_from_path
from PIL import Image

# ===================== CONFIG =====================
API_KEY = "K86291774288957"

OCR_PAYLOAD = {
    "apikey": API_KEY,
    "language": "eng",
    "ocrengine": 3,
    "scale": True,
    "detectOrientation": True
}

# ===================== PDF → IMAGES =====================
def pdf_to_images(pdf_path, out_dir, dpi=200):
    os.makedirs(out_dir, exist_ok=True)
    pages = convert_from_path(pdf_path, dpi=dpi)

    image_paths = []
    for i, page in enumerate(pages, start=1):
        img_path = os.path.join(out_dir, f"page_{i}.jpg")
        page = page.convert("L")  # grayscale
        page.save(img_path, "JPEG", quality=85)
        image_paths.append(img_path)

    return image_paths

# ===================== OCR =====================
def run_ocr(img_path, retries=3):
    for _ in range(retries):
        try:
            with open(img_path, "rb") as f:
                r = requests.post(
                    "https://api.ocr.space/parse/image",
                    files={"file": f},
                    data=OCR_PAYLOAD,
                    timeout=90
                )
            data = r.json()
            if not data.get("IsErroredOnProcessing"):
                parsed = data.get("ParsedResults", [])
                if parsed:
                    return parsed[0].get("ParsedText", "").strip()
        except:
            time.sleep(5)
    return ""

# ===================== QUESTION PARSER =====================
import re

def extract_questions(text):
    """
    Best-effort extraction:
    Detects patterns like:
    1.
    1(a)
    2(b)
    """
    blocks = re.split(r'\n(?=\d+\s*[\(\.]?)', text)
    results = []

    for block in blocks:
        header = re.match(r'(\d+)\s*(\([a-z]\))?', block.strip(), re.I)
        if header:
            qno = header.group(1)
            subtype = header.group(2).replace("(", "").replace(")", "") if header.group(2) else ""
            body = block.replace(header.group(0), "").strip()

            results.append({
                "question_number": f"{qno}{'(' + subtype + ')' if subtype else ''}",
                "subtype": subtype,
                "question_text": "",
                "answer_text": body
            })

    return results

# ===================== MAIN PIPELINE =====================
def ocr_pdf_pipeline(pdf_name):
    base = os.path.splitext(pdf_name)[0]
    out_dir = f"{base}_output"
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n📄 Processing: {pdf_name}")

    images = pdf_to_images(pdf_name, out_dir)

    all_text = []
    structured_answers = []

    # START FROM PAGE 1 (AS REQUESTED)
    for idx, img in enumerate(images, start=1):
        print(f"OCR page {idx}")
        text = run_ocr(img)
        all_text.append(f"\n--- PAGE {idx} ---\n{text}")

        extracted = extract_questions(text)
        structured_answers.extend(extracted)

        time.sleep(2)

    # ===================== SAVE TXT =====================
    txt_path = os.path.join(out_dir, f"{base}_answers.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("\n".join(all_text))

    # ===================== SAVE JSON =====================
    json_path = os.path.join(out_dir, f"{base}_answers.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(structured_answers, f, indent=2, ensure_ascii=False)

    print("✅ DONE")
    print(f"📄 TXT  → {txt_path}")
    print(f"🧾 JSON → {json_path}")

# ===================== RUN =====================
for pdf in uploaded.keys():
    ocr_pdf_pipeline(pdf)



📄 Processing: Archit Parag Ethics Updated.pdf
OCR page 1
OCR page 2
OCR page 3
OCR page 4
OCR page 5
OCR page 6
OCR page 7
OCR page 8
OCR page 9
OCR page 10
OCR page 11
OCR page 12
OCR page 13
OCR page 14
OCR page 15
OCR page 16
OCR page 17
OCR page 18
✅ DONE
📄 TXT  → Archit Parag Ethics Updated_output/Archit Parag Ethics Updated_answers.txt
🧾 JSON → Archit Parag Ethics Updated_output/Archit Parag Ethics Updated_answers.json


In [ ]:
def pdf_to_images(pdf_path, out_dir, dpi=200):
    os.makedirs(out_dir, exist_ok=True)
    pages = convert_from_path(pdf_path, dpi=dpi)

    image_paths = []
    for i, page in enumerate(pages, start=1):
        img_path = f"{out_dir}/page_{i}.jpg"
        page = page.convert("L")  # grayscale
        page.save(img_path, "JPEG", quality=85, subsampling=0)
        image_paths.append(img_path)

    return image_paths

In [ ]:
def is_handwritten_page(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # Edge detection (handwriting has irregular strokes)
    edges = cv2.Canny(img, 50, 150)

    # Ratio of edge pixels
    edge_ratio = np.sum(edges > 0) / edges.size

    # Printed pages → very low edge density
    # Handwritten pages → higher edge density
    return edge_ratio > 0.015

In [ ]:
def find_first_answer_page(image_paths):
    for idx, img in enumerate(image_paths):
        if is_handwritten_page(img):
            return idx + 1  # page number (1-based)
    return None

In [ ]:
from requests.exceptions import ReadTimeout, ConnectionError

def run_ocr_on_image(img_path, retries=3):
    for attempt in range(retries):
        try:
            with open(img_path, "rb") as f:
                r = requests.post(
                    "https://api.ocr.space/parse/image",
                    files={"file": f},
                    data=OCR_PAYLOAD,
                    timeout=90
                )
            result = r.json()
            if not result.get("IsErroredOnProcessing"):
                return result
        except (ReadTimeout, ConnectionError):
            time.sleep(8)

    return {"IsErroredOnProcessing": True}

In [ ]:
def ocr_pdf_pipeline(pdf_name):
    base = os.path.splitext(pdf_name)[0]
    out_dir = f"{base}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(out_dir, exist_ok=True)

    images = pdf_to_images(pdf_name, out_dir)

    start_page = find_first_answer_page(images)
    if start_page is None:
        print("❌ No handwritten pages detected.")
        return

    print(f"✅ Detected first answer page: PAGE {start_page}")

    all_pages = []
    full_text = []

    for page_num in range(start_page, len(images) + 1):
        img = images[page_num - 1]
        print(f"OCR page {page_num}")

        result = run_ocr_on_image(img)

        text = "[OCR FAILED]"
        if not result.get("IsErroredOnProcessing"):
            parsed = result.get("ParsedResults", [])
            if parsed:
                text = parsed[0].get("ParsedText", "").strip()

        all_pages.append({
            "page_number": page_num,
            "text": text
        })

        full_text.append(f"\n--- PAGE {page_num} ---\n{text}")
        time.sleep(2)

    with open(f"{out_dir}/answer_text.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(full_text))

    with open(f"{out_dir}/answer_structured.json", "w", encoding="utf-8") as f:
        json.dump(all_pages, f, indent=2)

    print(f"\n📂 Saved output in: {out_dir}")

In [ ]:
for pdf in uploaded.keys():
    ocr_pdf_pipeline(pdf)

✅ Detected first answer page: PAGE 1
OCR page 1
OCR page 2
OCR page 3
OCR page 4
OCR page 5
OCR page 6
OCR page 7
OCR page 8
OCR page 9
OCR page 10
OCR page 11
OCR page 12
OCR page 13
OCR page 14
OCR page 15
OCR page 16
OCR page 17
OCR page 18
OCR page 19
OCR page 20
OCR page 21
OCR page 22
OCR page 23
OCR page 24
OCR page 25
OCR page 26
OCR page 27
OCR page 28
OCR page 29
OCR page 30
OCR page 31
OCR page 32

📂 Saved output in: VisionIAS Toppers Answer Booklet Dongre Archit Parag_20260207_174330
